# 05. Avaliar atribuição — mãe + endereço

Depois do [`04_atribuir_mae_endereco.ipynb`](04_atribuir_mae_endereco.ipynb)
(modelo `p ≥ 0,90` + resgate R1/R2/R3).

Avalia notas, qualidade, quem ficou de fora, amostra por faixa/regra e
números com lista ouro.

Entradas:
- `splink_predictions_mae_endereco.parquet` ([`02d`](02d_aplicar_splink_mae_endereco.ipynb))
- `splink_atribuicao_mae_endereco.parquet` ([`04`](04_atribuir_mae_endereco.ipynb))


In [1]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    LISTA_OURO_ARQUIVO,
    MAX_CENSOS_POR_CPF,
    SPLINK_ATRIBUICAO_MAE_ENDERECO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    TABELA_CENSO_LIMPA,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    cpf_norm_sql,
    drop_splink_temp_tables,
    get_connection,
    lista_ouro_colunas,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

PISO_MODELO = 0.90
PISO_RESGATE = 0.50
PISO_BAIXO = 0.25
TETO = MAX_CENSOS_POR_CPF

print_paths()
require_input(
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    label='SPLINK_PREDICTIONS_MAE_ENDERECO (rode o 02d antes)',
)
require_input(
    SPLINK_ATRIBUICAO_MAE_ENDERECO,
    label='SPLINK_ATRIBUICAO_MAE_ENDERECO (rode o 04_atribuir_mae_endereco antes)',
)
require_input(LISTA_OURO_ARQUIVO, label='LISTA_OURO')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA, TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
print(f'Faixas 04: modelo>={PISO_MODELO} | resgate>={PISO_RESGATE} | baixo>={PISO_BAIXO} | teto={TETO}')


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

In [2]:
_tipo = con.execute('''
SELECT table_type
FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'splink_predictions'
''').fetchone()
if _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')

con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS_MAE_ENDERECO}')
''')

# Nome distinto da TABLE atribuicao do 04 no mesmo DuckDB.
for _nome in ('atribuicao_aval', 'atribuicao_2', 'atribuicao'):
    _t = con.execute(f'''
    SELECT table_type
    FROM information_schema.tables
    WHERE table_schema = 'main' AND table_name = '{_nome}'
    ''').fetchone()
    if _t:
        _k = 'VIEW' if _t[0].upper() == 'VIEW' else 'TABLE'
        con.execute(f'DROP {_k} IF EXISTS {_nome}')

cols_atr = [
    r[0]
    for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{SPLINK_ATRIBUICAO_MAE_ENDERECO}') LIMIT 0"
    ).fetchall()
]
tem_faixa = 'faixa' in cols_atr
tem_regra = 'regra' in cols_atr
col_faixa = 'faixa' if tem_faixa else 'CAST(NULL AS VARCHAR) AS faixa'
col_regra = 'regra' if tem_regra else 'CAST(NULL AS VARCHAR) AS regra'

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao_aval AS
SELECT
    unique_id_censo,
    unique_id_cpf,
    match_probability,
    {col_faixa},
    {col_regra}
FROM read_parquet('{SPLINK_ATRIBUICAO_MAE_ENDERECO}')
''')

n_app = con.execute(f'''
SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''').fetchone()[0]
n_atrib = con.execute('SELECT COUNT(*) FROM atribuicao_aval').fetchone()[0]
n_pred = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]

print(f'Aplicação: {n_app:,}')
print(f'Atribuídos: {n_atrib:,} ({100.0 * n_atrib / n_app:.1f}%)')
print(f'Sem CPF: {n_app - n_atrib:,} ({100.0 * (n_app - n_atrib) / n_app:.1f}%)')
print(f'Pares no parquet: {n_pred:,}')
print('Colunas faixa/regra:', tem_faixa, tem_regra)


Aplicação: 6,079,030
Atribuídos: 4,304,328 (70.8%)
Sem CPF: 1,774,702 (29.2%)
Pares no parquet: 6,315,410
Colunas faixa/regra: True True


## Notas e regra dos atribuídos

Distribuição de `match_probability`, contagem por `faixa`/`regra` do 04, e
Censos por CPF.


In [3]:
display(con.execute("""
SELECT
    CASE
        WHEN match_probability >= 0.99 THEN 'p_ge_099'
        WHEN match_probability >= 0.90 THEN 'p_090_099'
        WHEN match_probability >= 0.75 THEN 'p_075_090'
        WHEN match_probability >= 0.50 THEN 'p_050_075'
        WHEN match_probability >= 0.25 THEN 'p_025_050'
        ELSE 'p_lt_025'
    END AS faixa_p,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao_aval
GROUP BY 1
ORDER BY 1 DESC
""").df())

display(con.execute("""
SELECT
    faixa_atr,
    regra_atr,
    n,
    ROUND(100.0 * n / SUM(n) OVER (), 1) AS pct
FROM (
    SELECT
        coalesce(faixa, '(sem)') AS faixa_atr,
        coalesce(regra, '(sem)') AS regra_atr,
        COUNT(*) AS n
    FROM atribuicao_aval
    GROUP BY 1, 2
)
ORDER BY
    CASE faixa_atr
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
        ELSE 9
    END,
    regra_atr
""").df())

display(con.execute("""
SELECT
    n_censos,
    COUNT(*) AS n_cpf,
    n_censos * COUNT(*) AS n_pares
FROM (
    SELECT unique_id_cpf, COUNT(*) AS n_censos
    FROM atribuicao_aval
    GROUP BY 1
)
GROUP BY 1
ORDER BY 1
""").df())


,faixa_p,n,pct
0,p_ge_099,3832798,89.0
1,p_090_099,297222,6.9
2,p_075_090,45766,1.1
3,p_050_075,57525,1.3
4,p_025_050,71017,1.6


,faixa_atr,regra_atr,n,pct
0,modelo,score,4130020,96.0
1,resgate,R1,62354,1.4
2,resgate,R2,20897,0.5
3,resgate,R3,20040,0.5
4,baixo,R1,62882,1.5
5,baixo,R2,8135,0.2


,n_censos,n_cpf,n_pares
0,1,4081752,4081752
1,2,108573,217146
2,3,1810,5430


## Qualidade dos pares atribuídos

No par que saiu: mãe, endereço e distância de ano de nascimento.


In [4]:
display(con.execute(f'''
SELECT
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
    ) / COUNT(*), 1) AS pct_mae_preenchida,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
          AND ca.nome_mae_phon = pb.nome_mae_phon
    ) / COUNT(*), 1) AS pct_mae_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
          AND ca.logradouro_norm = pb.logradouro_norm
    ) / COUNT(*), 1) AS pct_logradouro_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
    ) / COUNT(*), 1) AS pct_cep_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento = pb.data_nascimento
    ) / COUNT(*), 1) AS pct_data_igual
FROM atribuicao_aval a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
''').df())

display(con.execute(f'''
SELECT
    CASE
        WHEN ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL THEN 'sem_data'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) = 0 THEN 'delta_0'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 1 THEN 'delta_1'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 5 THEN 'delta_2_5'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 10 THEN 'delta_6_10'
        ELSE 'delta_gt_10'
    END AS faixa_delta_ano,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao_aval a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
GROUP BY 1
ORDER BY 1
''').df())


,n,pct_mae_preenchida,pct_mae_igual,pct_logradouro_igual,pct_cep_igual,pct_data_igual
0,4304328,41.6,24.1,25.2,65.7,83.1


,faixa_delta_ano,n,pct
0,delta_0,3759455,87.3
1,delta_1,127662,3.0
2,delta_2_5,49583,1.2
3,delta_6_10,30031,0.7
4,delta_gt_10,11228,0.3
5,sem_data,326369,7.6


## Quem ficou de fora

Melhor par no parquet por Censo sem atribuição. Quem tinha `p ≥ 0,50` e
mesmo assim não saiu: teto ou falhou o resgate (R1/R2/R3).


In [5]:
# Mesmo desempate do 04: nota → mãe preenchida → logradouro → CEP.
con.execute(f'''
CREATE OR REPLACE TABLE melhor_qualquer AS
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.unique_id_censo
    ORDER BY
        p.match_probability DESC,
        CAST(
            ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
            AS INTEGER
        ) DESC,
        CAST(
            ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
            AND ca.logradouro_norm = pb.logradouro_norm
            AS INTEGER
        ) DESC,
        CAST(
            ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
            AS INTEGER
        ) DESC,
        p.unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_piso AS
SELECT *
FROM melhor_qualquer
WHERE match_probability >= {PISO_RESGATE}
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id AS unique_id_censo
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
    SELECT 1 FROM atribuicao_aval a WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute(f'''
CREATE OR REPLACE TABLE sem_cpf_melhor AS
SELECT
    s.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    CASE
        WHEN m.match_probability IS NULL THEN 'sem_par'
        WHEN m.match_probability >= 0.50 THEN 'p_ge_050'
        WHEN m.match_probability >= 0.25 THEN 'p_025_050'
        WHEN m.match_probability >= 0.05 THEN 'p_005_025'
        ELSE 'p_lt_005'
    END AS faixa_p
FROM censo_sem_cpf s
LEFT JOIN melhor_qualquer m ON m.unique_id_censo = s.unique_id_censo
''')

n_sem = con.execute('SELECT COUNT(*) FROM censo_sem_cpf').fetchone()[0]
assert n_atrib + n_sem == n_app

display(con.execute(f'''
SELECT
    a.faixa_p,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / {n_sem}, 1) AS pct_sem_cpf,
    ROUND(100.0 * coalesce(g.n, 0) / {n_app}, 1) AS pct_aplicacao
FROM (VALUES
    (1, 'p_ge_050'),
    (2, 'p_025_050'),
    (3, 'p_005_025'),
    (4, 'p_lt_005'),
    (5, 'sem_par')
) a(ord, faixa_p)
LEFT JOIN (
    SELECT faixa_p, COUNT(*) AS n
    FROM sem_cpf_melhor
    GROUP BY 1
) g ON g.faixa_p = a.faixa_p
ORDER BY a.ord
''').df())

# Censos com p >= PISO no melhor-par, mas CPF estourou o teto.
con.execute(f'''
CREATE OR REPLACE TABLE cpf_n_melhor_piso AS
SELECT unique_id_cpf, COUNT(*) AS n_censos
FROM melhor_piso
GROUP BY 1
''')

n_teto = con.execute(f'''
SELECT COUNT(*)
FROM sem_cpf_melhor s
JOIN cpf_n_melhor_piso c ON c.unique_id_cpf = s.unique_id_cpf
WHERE s.match_probability >= {PISO_RESGATE}
  AND c.n_censos > {TETO}
''').fetchone()[0]
n_alto = con.execute(f'''
SELECT COUNT(*) FROM sem_cpf_melhor
WHERE match_probability >= {PISO_RESGATE}
''').fetchone()[0]
print(f'Sem CPF com melhor p >= {PISO_RESGATE}: {n_alto:,}')
print(f'  dos quais teto (CPF com >{TETO} no melhor-par): {n_teto:,}')


,faixa_p,n,pct_sem_cpf,pct_aplicacao
0,p_ge_050,119281,6.7,2.0
1,p_025_050,132252,7.5,2.2
2,p_005_025,242856,13.7,4.0
3,p_lt_005,0,0.0,0.0
4,sem_par,1280313,72.1,21.1


Sem CPF com melhor p >= 0.5: 119,281
  dos quais teto (CPF com >3 no melhor-par): 386


## Perfil dos 1,28M sem par

Compara Censos **atribuídos** × **sem par** no predict (e o total sem CPF).
Pergunta: falta data? nome frequente? poucos/muitos tokens? sem mãe? sem endereço?


In [6]:
# Top 10 / top 50 do primeiro nome no Censo de aplicação (referência de "nome frequente").
con.execute(f'''
CREATE OR REPLACE TABLE top_primeiro AS
SELECT
    primeiro_nome_phon,
    COUNT(*) AS n,
    ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC, primeiro_nome_phon) AS rk
FROM {SPLINK_INPUT_VIEW}
WHERE origem = 'censo'
  AND primeiro_nome_phon IS NOT NULL
GROUP BY 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_perfil AS
SELECT
    c.unique_id AS unique_id_censo,
    CASE
        WHEN a.unique_id_censo IS NOT NULL THEN 'atribuido'
        WHEN s.faixa_p = 'sem_par' THEN 'sem_par'
        WHEN s.unique_id_censo IS NOT NULL THEN 'sem_cpf_com_par'
        ELSE 'outro'
    END AS grupo,
    c.data_nascimento IS NULL AS sem_data,
    c.primeiro_nome_phon IS NULL AS sem_primeiro,
    c.ultimo_nome_phon IS NULL AS sem_ultimo,
    c.nome_completo_phon IS NULL AS sem_nome,
    c.nome_mae_phon IS NULL AS sem_mae,
    c.cep IS NULL AS sem_cep,
    c.logradouro_norm IS NULL AS sem_logradouro,
    c.cod_municipio IS NULL AS sem_municipio,
    c.sexo IS NULL AS sem_sexo,
    coalesce(len(string_split(c.nome_completo_phon, ' ')), 0) AS n_tokens,
    coalesce(t.rk <= 10, FALSE) AS primeiro_top10,
    coalesce(t.rk <= 50, FALSE) AS primeiro_top50,
    (
        c.data_nascimento IS NULL
        AND c.nome_mae_phon IS NULL
        AND (c.cep IS NULL OR c.logradouro_norm IS NULL)
    ) AS pobre_data_mae_end
FROM {SPLINK_INPUT_VIEW} c
LEFT JOIN atribuicao_aval a ON a.unique_id_censo = c.unique_id
LEFT JOIN sem_cpf_melhor s ON s.unique_id_censo = c.unique_id
LEFT JOIN top_primeiro t ON t.primeiro_nome_phon = c.primeiro_nome_phon
WHERE c.origem = 'censo'
''')

print('Top 10 primeiro_nome_phon (aplicação):')
display(con.execute('''
SELECT rk, primeiro_nome_phon, n
FROM top_primeiro
WHERE rk <= 10
ORDER BY rk
''').df())

# Taxas por grupo (atribuído × sem_par × sem_cpf_com_par).
display(con.execute('''
SELECT
    grupo,
    COUNT(*) AS n,
    ROUND(100.0 * AVG(sem_data::INT), 1) AS pct_sem_data,
    ROUND(100.0 * AVG(sem_primeiro::INT), 1) AS pct_sem_primeiro,
    ROUND(100.0 * AVG(sem_ultimo::INT), 1) AS pct_sem_ultimo,
    ROUND(100.0 * AVG(sem_nome::INT), 1) AS pct_sem_nome,
    ROUND(100.0 * AVG(sem_mae::INT), 1) AS pct_sem_mae,
    ROUND(100.0 * AVG(sem_cep::INT), 1) AS pct_sem_cep,
    ROUND(100.0 * AVG(sem_logradouro::INT), 1) AS pct_sem_logradouro,
    ROUND(100.0 * AVG(sem_municipio::INT), 1) AS pct_sem_municipio,
    ROUND(100.0 * AVG(sem_sexo::INT), 1) AS pct_sem_sexo,
    ROUND(100.0 * AVG(primeiro_top10::INT), 1) AS pct_primeiro_top10,
    ROUND(100.0 * AVG(primeiro_top50::INT), 1) AS pct_primeiro_top50,
    ROUND(100.0 * AVG(pobre_data_mae_end::INT), 1) AS pct_pobre_data_mae_end
FROM censo_perfil
WHERE grupo IN ('atribuido', 'sem_par', 'sem_cpf_com_par')
GROUP BY 1
ORDER BY
    CASE grupo
        WHEN 'atribuido' THEN 1
        WHEN 'sem_cpf_com_par' THEN 2
        WHEN 'sem_par' THEN 3
    END
''').df())

# Distribuição de tokens do nome.
display(con.execute('''
SELECT
    grupo,
    CASE
        WHEN n_tokens <= 1 THEN '1'
        WHEN n_tokens = 2 THEN '2'
        WHEN n_tokens = 3 THEN '3'
        WHEN n_tokens = 4 THEN '4'
        ELSE '5+'
    END AS faixa_tokens,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY grupo), 1) AS pct
FROM censo_perfil
WHERE grupo IN ('atribuido', 'sem_par')
GROUP BY 1, 2
ORDER BY
    CASE grupo WHEN 'atribuido' THEN 1 ELSE 2 END,
    faixa_tokens
''').df())

# Combinações que matam blocking (sem_par): quantos têm pelo menos um "buraco".
display(con.execute('''
SELECT
    ROUND(100.0 * COUNT(*) FILTER (WHERE sem_data) / COUNT(*), 1) AS pct_sem_data,
    ROUND(100.0 * COUNT(*) FILTER (WHERE sem_mae) / COUNT(*), 1) AS pct_sem_mae,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE sem_cep AND sem_logradouro
    ) / COUNT(*), 1) AS pct_sem_cep_e_logr,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE sem_data OR sem_primeiro OR sem_ultimo
    ) / COUNT(*), 1) AS pct_furo_nome_ou_data,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE sem_data
          AND sem_mae
          AND (sem_cep OR sem_logradouro)
    ) / COUNT(*), 1) AS pct_sem_data_mae_e_end,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE NOT sem_data
          AND NOT sem_primeiro
          AND NOT sem_ultimo
          AND NOT sem_mae
          AND NOT sem_cep
          AND NOT sem_logradouro
    ) / COUNT(*), 1) AS pct_campos_cheios_mesmo_assim
FROM censo_perfil
WHERE grupo = 'sem_par'
''').df())


Top 10 primeiro_nome_phon (aplicação):


,rk,primeiro_nome_phon,n
0,1,MARIA,517158
1,2,JOZI,202433
2,3,ANA,141590
3,4,ANTONIU,134041
4,5,JOAU,121617
5,6,FRANSISKU,115200
6,7,RAIMUNDU,81643
7,8,LUIS,66276
8,9,KARLUS,58004
9,10,PEDRU,53592


,grupo,n,pct_sem_data,pct_sem_primeiro,pct_sem_ultimo,pct_sem_nome,pct_sem_mae,pct_sem_cep,pct_sem_logradouro,pct_sem_municipio,pct_sem_sexo,pct_primeiro_top10,pct_primeiro_top50,pct_pobre_data_mae_end
0,atribuido,4304328,7.6,0.0,0.0,0.0,58.4,9.9,0.0,0.0,0.0,24.9,38.0,0.1
1,sem_cpf_com_par,494389,38.5,0.0,0.0,0.0,90.4,13.5,0.0,0.0,0.0,32.2,44.0,4.9
2,sem_par,1280313,36.2,0.0,2.0,0.0,72.3,14.4,0.0,0.0,0.0,20.3,31.7,4.3


,grupo,faixa_tokens,n,pct
0,atribuido,1,292,0.0
1,atribuido,2,251030,5.8
2,atribuido,3,2270346,52.7
3,atribuido,4,1675551,38.9
4,atribuido,5+,107109,2.5
5,sem_par,1,25080,2.0
6,sem_par,2,383974,30.0
7,sem_par,3,571734,44.7
8,sem_par,4,278554,21.8
9,sem_par,5+,20971,1.6


,pct_sem_data,pct_sem_mae,pct_sem_cep_e_logr,pct_furo_nome_ou_data,pct_sem_data_mae_e_end,pct_campos_cheios_mesmo_assim
0,36.2,72.3,0.0,36.4,4.3,16.9


## Pares aleatórios por faixa / regra

Amostra nova a cada execução (`ORDER BY random()`). Ajuste `N_AMOSTRA`.


In [7]:
N_AMOSTRA = 5

cols_par = '''
    a.match_probability,
    a.faixa,
    a.regra,
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ca.nome_mae AS mae_censo,
    pb.nome_mae AS mae_cpf,
    ca.logradouro_norm AS logradouro_censo,
    pb.logradouro_norm AS logradouro_cpf,
    ca.cep AS cep_censo,
    pb.cep AS cep_cpf,
    a.unique_id_censo,
    a.unique_id_cpf
'''

print('--- Por faixa de nota (atribuídos) ---')
for nome, lo, hi in [
    ('p_ge_099', 0.99, 1.01),
    ('p_090_099', 0.90, 0.99),
    ('p_050_090', 0.50, 0.90),
    ('p_025_050', 0.25, 0.50),
]:
    print(f'\n=== {nome} [{lo}, {hi}) ===')
    display(con.execute(f'''
    SELECT {cols_par}
    FROM atribuicao_aval a
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
    WHERE a.match_probability >= {lo}
      AND a.match_probability < {hi}
    ORDER BY random()
    LIMIT {N_AMOSTRA}
    ''').df())

print('\n--- Por regra do 04 ---')
for regra in ['score', 'R1', 'R2', 'R3']:
    print(f'\n=== regra={regra} ===')
    display(con.execute(f'''
    SELECT {cols_par}
    FROM atribuicao_aval a
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
    WHERE a.regra = '{regra}'
    ORDER BY random()
    LIMIT {N_AMOSTRA}
    ''').df())

print('\n--- Sem CPF (melhor par), faixas abaixo do modelo ---')
for nome, lo, hi in [
    ('p_050_090', 0.50, 0.90),
    ('p_025_050', 0.25, 0.50),
    ('p_005_025', 0.05, 0.25),
]:
    print(f'\n=== {nome} [{lo}, {hi}) ===')
    display(con.execute(f'''
    SELECT
        s.match_probability,
        ca.nome_completo AS nome_censo,
        pb.nome_completo AS nome_cpf,
        ca.data_nascimento AS dob_censo,
        pb.data_nascimento AS dob_cpf,
        ca.nome_mae AS mae_censo,
        pb.nome_mae AS mae_cpf,
        ca.logradouro_norm AS logradouro_censo,
        pb.logradouro_norm AS logradouro_cpf,
        s.unique_id_censo,
        s.unique_id_cpf
    FROM sem_cpf_melhor s
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = s.unique_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = s.unique_id_cpf
    WHERE s.match_probability >= {lo}
      AND s.match_probability < {hi}
    ORDER BY random()
    LIMIT {N_AMOSTRA}
    ''').df())


--- Por faixa de nota (atribuídos) ---

=== p_ge_099 [0.99, 1.01) ===


,match_probability,faixa,regra,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,cep_censo,cep_cpf,unique_id_censo,unique_id_cpf
0,0.999411,modelo,score,ANA BEATRIZ CONCEICAO ALVES SOUSA,ANA BEATRIZ CONCEICAO ALVES SOUSA,2010-10-19,2010-10-19,None,MARIA ANTONIA CONCEICAO,SERGIPE,D,65919180,65917080,censo_2105302050002370050010000116000001000000006,cpf_63285446340
1,0.999036,modelo,score,FRANCISCO CHAGAS SOUSA SANTOS FILHO,FRANCISCO CHAGAS SOUSA SANTOS FILHO,1977-12-15,1977-12-15,None,RAIMUNDA SOUSA SANTOS,QUINZE,BOM JESUS,None,65900000,censo_2105302050003810290020000294000001000000001,cpf_92880100330
2,0.996976,modelo,score,CLEOMAR MILHOMEM SOUSA,CLEOMAR MILHOMEM SOUSA,1956-12-06,1955-12-06,None,ROSINHA MILHOMEM SOUSA,PARA O BAIXAO DO CEDRO,POVOADO BAIXAO DO CEGO S N,65768000,65768000,censo_2109759050000270000110000196000001000000002,cpf_21370176287
3,1.000000,modelo,score,MARCOS RONALD SILVA COSTA,MARCOS RONALD SILVA COSTA,2007-05-21,2007-05-21,GENELDA DUTRA SILVA,GENELDA DUTRA SILVA,SEIS,06,None,65000000,censo_2111300050007370060010000221000001000000002,cpf_61715613392
4,0.999431,modelo,score,EDENILTON RIBEIRO SARAIVA,EDENILTON RIBEIRO SARAIVA,1989-10-30,1989-10-30,None,MARIA ROSARIO RIBEIRO,RIO PERITORO,SEIS UNID 101,65094011,65000000,censo_2111300050013100050010000257000001000000002,cpf_63086313374



=== p_090_099 [0.9, 0.99) ===


,match_probability,faixa,regra,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,cep_censo,cep_cpf,unique_id_censo,unique_id_cpf
0,0.942160,modelo,score,ANTONIA CONCEICAO SILVA,ANTONIA CONCEICAO SILVA,1987-05-27,1987-05-27,None,CUSTODIA CONCEICAO SILVA,JOAO AMORIM,POVOADO BREJO GRANDE,65320000,65716000,censo_2113009050000560030020000108000003000000002,cpf_02404685384
1,0.973038,modelo,score,DEOSDETE SILVA SANTOS,DEUSDETE SILVA SANTOS,1964-11-16,1964-11-16,None,MARIA JESUS SILVA,SANHARO 2,RURAL,65610000,65609899,censo_2100303050000410000130000181000001000000001,cpf_00899175376
2,0.980499,modelo,score,MARIA FATIMA SILVA OLIVEIRA,MARIA FATIMA SILVA CONCEICAO OLIVEIRA,1966-12-05,1965-12-05,None,ANTONIA SILVA CONCEICAO,DO FIO,DO FIO,65415000,65415000,censo_2103604050000040020060000350000001000000001,cpf_00742692329
3,0.951257,modelo,score,SILVAN CARLOS SANTOS CUNHA,SILVAN CARLOS SANTOS CUNHA,None,1993-06-04,None,LUCILENE SILVA SANTOS,SEM DENOMINACAO,PROJETADA,65278000,65278000,censo_2112407050000060010020000034000001000000003,cpf_07720185374
4,0.956806,modelo,score,ACELINO OLIVEIRA SILVA,ARCELINO OLIVEIRA SILVA,1985-05-17,1985-05-17,None,MARIA GRACAS OLIVEIRA SILVA,DA QUADRA,DA MANGUEIRAO,65948000,65940000,censo_2105351050000150110010000321000001000000001,cpf_60145516385



=== p_050_090 [0.5, 0.9) ===


...

,match_probability,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,unique_id_censo,unique_id_cpf
0,0.517178,LUCAS SANTOS COSTA,LUCAS SANTOS COSTA,1998-08-12,2020-06-14,None,LEIDIANE PEREIRA SANTOS,BENJAMIN CONSTANT,POVOADO CANTO DO SACO,censo_2100907050000990200020000180000001000000003,cpf_11376540312
1,0.651291,FRANCISCO ASSIS FREITAS MELONIO FREITAS MELONIO,FRANCISCO ASSIS FREITAS MELONIO,None,1980-07-04,None,BRASILISE JESUS FREITAS MELONIO,EDMILSON GONCALVES,EDIMILSON GONCALVES,censo_2108256050000370020030000161000001000000002,cpf_83323120353
2,0.746288,LUIS LIMA,LUIS LIMA,None,1952-09-07,None,MARIA CONCEICAO LIMA,TRAVASSOS FURTADO,DA ESTRELA,censo_2110005050000020100020000212000001000000001,cpf_09380019220
3,0.728452,OZILA DIAS CARNEIRO SOUSA,OZILA DIAS CARNEIRO SOUSA,1940-07-03,1948-07-03,None,CANUTA MARIA CONCEICAO,PRINCIPAL,SAO FRANCISCO,censo_2105922050000240000210000152000002000000002,cpf_49941321353
4,0.766780,RONILSO RIBEIRO MENDONCA,RONILSON RIBEIRO MENDONCA,1991-07-27,1991-07-27,None,JURANEIDE RIBEIRO MENDONCA,SOARES MELO,CASTELO BRANCO,censo_2108108050000050040040000443000001000000001,cpf_04677181306



=== p_025_050 [0.25, 0.5) ===


,match_probability,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,unique_id_censo,unique_id_cpf
0,0.289340,DAIANA CRISTINA SILVA SOARES,DAYANA CRISTINA SILVA SOARES,None,1991-02-05,None,SILVANA CRISTINA JESUS SILVA SOARES,SENADOR JOSE SARNEY,PRINCIPAL,censo_2111300050013810130030000354000001000000006,cpf_05403142347
1,0.255947,ERISVAN GASPAR CONCEICAO,ERISVAN GASPAR CONCEICAO,2005-02-28,2005-02-23,None,MARIA CONCEICAO GASPAR OLIVEIRA,PARA COCAL,1 DE JULHO,censo_2103174050000380000150000234000001000000002,cpf_10679821325
2,0.373542,EVERTON THAEYGO SILVA,EWERTON THAYEGO SILVA,2007-05-10,2007-05-10,None,JARDEANE BASILIO SILVA,ANASTACIO DE SOUSA LIMA,TRES,censo_2112209050000420040020000155000001000000002,cpf_07791074389
3,0.395623,ANTONIO JESUS SOUSA,ANTONIO JESUS SOUSA,1995-06-18,1963-02-08,None,MARIA ROSA JESUS,PRINCIPAL,DA ALEGRIA,censo_2100477050000140060010000099000001000000003,cpf_81042680310
4,0.440588,ANA KELLY,ANA KELLEN CASTRO ALBUQUERQUE,1997-07-29,1997-07-29,ANA CONCEICAO,ANA CONCEICAO CASTRO,22,SAO JOSE,censo_2101202050001350060020000265000001000000002,cpf_01915503370



=== p_005_025 [0.05, 0.25) ===


,match_probability,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,unique_id_censo,unique_id_cpf
0,0.077442,MARIA CRISTINA MORAES OLIVEIRA,MARIA CRISTINA MORAIS OLIVEIRA,1971-02-15,1981-10-21,None,ROZILDA OLIVEIRA LIMA,SANTA ROSA,DA BALANCA,censo_2108256050000210001620000149000002000000002,cpf_05169209363
1,0.063545,ALINE SANTOS RODRIGUES,ALINE SANTOS RODRIGUES,None,2002-11-05,None,NIRACI RODRIGUES SANTOS,NOVA,GRANDE,censo_2111300050005840020030000084000001000000004,cpf_60889120307
2,0.140982,LUCAS BENJAMIM COSTA SILVA,LUCAS BENJAMIM COSTA BRITO,2006-06-22,2006-06-22,None,LUCILENE SILVA COSTA BRITO,ZINOR CALDEIRAS,SINOR CALDEIRAS,censo_2108207050000670210010000079000001000000003,cpf_61378365330
3,0.073601,ANTONIO VALERIO PEREIRA SANTOS,ANTONIO VALERIO PEREIRA SANTOS,None,1938-07-22,None,MARIA PEREIRA SANTOS,MADA ALEGRE,PREGO,censo_2103000050001400000890000319000001000000001,cpf_64094634304
4,0.050301,EDER SANTOS,EDER SANTOS,None,1900-11-29,None,DAMIANA LOBATO,JOSE BONIFACIO,SAO LUIS REI DE FRANCA,censo_2109403050000230190020000119000001000000001,cpf_00057568391


## Observação sobre piso

O 04 já resgata `0,25 ≤ p < 0,90` com R1/R2/R3. Baixar o piso do **modelo**
(`p ≥ 0,90`) sem regra tende a puxar falso positivo — use a amostra acima
antes de afrouxar.


In [8]:
display(con.execute(f"""
SELECT
    faixa_atr,
    n,
    ROUND(100.0 * n / SUM(n) OVER (), 1) AS pct_atrib,
    ROUND(100.0 * n / {n_app}, 1) AS pct_aplicacao
FROM (
    SELECT
        coalesce(faixa, '(sem)') AS faixa_atr,
        COUNT(*) AS n
    FROM atribuicao_aval
    GROUP BY 1
)
ORDER BY
    CASE faixa_atr
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
        ELSE 9
    END
""").df())


,faixa_atr,n,pct_atrib,pct_aplicacao
0,modelo,4130020,96.0,67.9
1,resgate,103291,2.4,1.7
2,baixo,71017,1.6,1.2


## Lista ouro (censo_limpo)

Universo com a lista ainda dentro. Quem está na ouro e no pareamento conta
como `lista_ouro`. `nos_dois` tem de ser zero (essa gente saiu da aplicação
antes do 04).


In [9]:
ouro_cols = {
    r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{LISTA_OURO_ARQUIVO}') LIMIT 0"
    ).fetchall()
}
id_ouro, cpf_ouro = lista_ouro_colunas(ouro_cols)

con.execute(f'''
CREATE OR REPLACE TABLE lista_ouro AS
SELECT
    CAST({id_ouro} AS VARCHAR) AS id_censo,
    {cpf_norm_sql(cpf_ouro)} AS cpf_norm
FROM read_parquet('{LISTA_OURO_ARQUIVO}')
WHERE {id_ouro} IS NOT NULL AND {cpf_ouro} IS NOT NULL
''')

display(con.execute(f'''
SELECT
    a.origem,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / nullif(t.n_censo, 0), 1) AS pct
FROM (VALUES
    (1, 'lista_ouro'),
    (2, 'pareamento'),
    (3, 'sem_cpf')
) a(ord, origem)
CROSS JOIN (
    SELECT COUNT(*) AS n_censo FROM {TABELA_CENSO_LIMPA}
) t
LEFT JOIN (
    SELECT
        CASE
            WHEN o.id_censo IS NOT NULL THEN 'lista_ouro'
            WHEN p.unique_id_censo IS NOT NULL THEN 'pareamento'
            ELSE 'sem_cpf'
        END AS origem,
        COUNT(*) AS n
    FROM {TABELA_CENSO_LIMPA} c
    LEFT JOIN (
        SELECT DISTINCT id_censo FROM lista_ouro
    ) o ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
    LEFT JOIN (
        SELECT DISTINCT unique_id_censo FROM atribuicao_aval
    ) p ON p.unique_id_censo = c.unique_id
    GROUP BY 1
) g ON g.origem = a.origem
ORDER BY a.ord
''').df())

n_ambos = con.execute(f'''
SELECT COUNT(*)
FROM {TABELA_CENSO_LIMPA} c
JOIN (SELECT DISTINCT id_censo FROM lista_ouro) o
  ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
JOIN (SELECT DISTINCT unique_id_censo FROM atribuicao_aval) p
  ON p.unique_id_censo = c.unique_id
''').fetchone()[0]
print('nos_dois:', f'{n_ambos:,}')
assert n_ambos == 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,origem,n,pct
0,lista_ouro,510774,7.8
1,pareamento,4304328,65.3
2,sem_cpf,1774702,26.9


nos_dois: 0


In [10]:
con.close()
print('Fim do 05.')


Fim do 05.
